# 01 · AI4Privacy converted-dataset exploration

**Purpose.** Audit the *existing converted files* supplied with ConsentShield. This notebook does not download or alter the source corpus and does not assume the original AI4Privacy file is present.

**Inputs** `data/ai4privacy_converted/{train,validation,test}_gliner.jsonl`.

**Outputs** `results/ai4privacy_exploration.json`.

**Schema used by this dataset:** `tokenized_text` is a list of tokens; `ner` contains inclusive token-index spans `[start_token, end_token, label]`; `ner_labels` is the row's label inventory; `unmapped_span_count` records source annotations not converted. The source labels have already been converted, so this notebook cannot recover or independently inspect dropped original label names.

In [ ]:
import json, platform, collections
from pathlib import Path

# Works when launched from the project root or its notebooks/ directory.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents]
             if (p / "data" / "ai4privacy_converted").is_dir()), HERE)
DATA = ROOT / "data" / "ai4privacy_converted"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
SPLITS = {"train": DATA / "train_gliner.jsonl",
          "validation": DATA / "validation_gliner.jsonl",
          "test": DATA / "test_gliner.jsonl"}
missing = [str(p) for p in SPLITS.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing converted dataset file(s): " + ", ".join(missing))
print("Project root:", ROOT)
print("Machine:", platform.machine())
print("Auditing converted files only; no source dataset is downloaded.")

## Dataset audit

The scan is streaming to keep memory use bounded. Span indices are checked against token counts; this is a structural check, not a judgement that the source annotations are semantically correct.

In [ ]:
def audit_jsonl(path):
    rows = 0
    positives = 0
    empty_text = 0
    bad_spans = 0
    spans = collections.Counter()
    languages = collections.Counter()
    source_counts = collections.Counter()
    unmapped_rows = 0
    unmapped_total = 0
    token_lengths = []
    for line_no, line in enumerate(path.open(encoding="utf-8"), 1):
        if not line.strip():
            continue
        r = json.loads(line)
        rows += 1
        tokens = r.get("tokenized_text") or []
        ner = r.get("ner") or []
        if not tokens:
            empty_text += 1
        if ner:
            positives += 1
        token_lengths.append(len(tokens))
        languages[r.get("language", "UNKNOWN")] += 1
        source_counts[r.get("source", "UNKNOWN")] += 1
        unmapped = int(r.get("unmapped_span_count", 0) or 0)
        unmapped_total += unmapped
        unmapped_rows += int(unmapped > 0)
        for span in ner:
            if not isinstance(span, (list, tuple)) or len(span) != 3:
                bad_spans += 1
                continue
            try:
                start, end, label = int(span[0]), int(span[1]), str(span[2])
            except (TypeError, ValueError):
                bad_spans += 1
                continue
            if not (0 <= start <= end < len(tokens)):
                bad_spans += 1
                continue
            spans[label] += 1
    token_lengths.sort()
    def pct(q):
        if not token_lengths: return 0
        return token_lengths[min(len(token_lengths)-1, int((len(token_lengths)-1)*q))]
    return {
        "file": str(path.relative_to(ROOT)), "rows": rows,
        "rows_with_spans": positives, "rows_without_spans": rows - positives,
        "empty_token_lists": empty_text,
        "span_count_by_label": dict(spans), "total_spans": int(sum(spans.values())),
        "invalid_span_count": bad_spans,
        "unmapped_rows": unmapped_rows, "unmapped_span_count": unmapped_total,
        "language_counts": dict(languages), "source_counts": dict(source_counts),
        "token_count_p50": pct(0.50), "token_count_p95_approx": pct(0.95),
    }

report = {"machine": platform.machine(), "splits": {k: audit_jsonl(v) for k, v in SPLITS.items()}}
for split, item in report["splits"].items():
    print(f"\n{split}: {item['rows']:,} rows | {item['total_spans']:,} mapped spans | "
          f"{item['unmapped_span_count']:,} unmapped spans | {item['invalid_span_count']} invalid spans")
    print("  labels:", item["span_count_by_label"])
    print("  languages:", item["language_counts"])
print("\nNote: labels are those retained by conversion. Unmapped source annotations are counted, but their original label names are not present here.")

In [ ]:
out = RESULTS / "ai4privacy_exploration.json"
out.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved", out)